# Step 05.2 — Minimal Calculus for Learning

这一节只建立训练真正需要的最小微积分直觉：

1. 导数 = 局部斜率 / 局部敏感度；
2. 用 finite difference 近似导数；
3. 看懂几个最常用的标量导数；
4. 第一次使用 chain rule，把 parameter 和 loss 连起来。

暂时不碰向量导数、Jacobian 或矩阵微积分。

## 1. 从“斜率”开始：一个输入变一点，输出变多少？

先看最简单函数：

```text
f(x) = x²
```

在 `x=3`：

```text
f(3)=9
```

如果把 x 稍微增加一点，例如 `3 -> 3.1`：

```text
f(3.1)=9.61
```

x 增加了 `0.1`，f 增加了 `0.61`。

于是这一小段的平均变化率：

```text
0.61 / 0.1 = 6.1
```

如果我们让这一步越来越小，这个变化率会越来越接近 `6`。

这就是 `f(x)=x²` 在 `x=3` 的导数。

In [ ]:
def f(x):
    return x * x

x = 3.0

for h in [1.0, 0.1, 0.01, 0.001]:
    slope = (f(x + h) - f(x)) / h
    print(f"h={h:<6} approximate slope={slope}")

你应该看到结果逐渐接近：

```text
6
```

这个公式：

```text
(f(x+h)-f(x)) / h
```

叫 finite difference（有限差分）。

它不是神经网络训练最终使用的求导方式，但非常适合建立直觉：

> 导数告诉我们：在当前位置，输入稍微变化，输出大约以多快的速度变化。

## 2. 导数符号告诉我们“往哪边走”

仍然看 `f(x)=x²`。

它的导数是：

```text
f'(x)=2x
```

所以：

```text
x=3  -> f'(3)= 6
x=-3 -> f'(-3)=-6
x=0  -> f'(0)= 0
```

含义：

- 导数 > 0：x 往右增加时，函数局部会上升；
- 导数 < 0：x 往右增加时，函数局部会下降；
- 导数 = 0：当前位置一阶上比较平坦。

以后训练模型时，我们最关心的是 loss 对 parameter 的导数，因为它告诉我们 parameter 往哪个方向改会让 loss 下降。

## 3. 现在只记几个最小导数规则

当前阶段只需要：

```text
d/dx (constant) = 0
d/dx (x)        = 1
d/dx (x²)       = 2x
d/dx (a*x+b)    = a
```

例如：

```text
f(x)=5x+3
f'(x)=5
```

意思是无论 x 在哪里，x 每增加一点，f 大约以 5 倍速度变化。

## 4. 第一次把导数接到“模型参数 -> prediction -> loss”

构造一个只有一个 parameter 的超小模型：

```text
prediction = w * x
```

固定：

```text
x = 2
target = 10
```

于是：

```text
prediction = 2w
loss = (2w - 10)²
```

现在训练的核心问题变成：

> w 稍微变化时，loss 会怎么变化？

In [ ]:
def loss_from_w(w):
    x = 2.0
    target = 10.0
    prediction = w * x
    return (prediction - target) ** 2

w = 3.0
h = 0.001

approx_derivative = (loss_from_w(w + h) - loss_from_w(w)) / h

print("loss at w=3:", loss_from_w(w))
print("finite-difference derivative:", approx_derivative)

当 `w=3`：

```text
prediction = 3*2 = 6
target = 10
loss = (6-10)² = 16
```

finite difference 会告诉你导数大约是：

```text
-16
```

负号非常重要：

> 在 w=3 附近，如果把 w 稍微增大，loss 会下降。

所以如果目标是减小 loss，我们应该让 w 往更大的方向走。

## 5. Chain Rule：为什么中间隔着 prediction 仍然能算？

真正的数据流不是 `w -> loss` 一步完成，而是：

```text
w
↓
prediction = w*x
↓
error = prediction-target
↓
loss = error²
```

为了让符号更清楚，设：

```text
e = wx - target
L = e²
```

我们知道：

```text
dL/de = 2e
de/dw = x
```

chain rule 告诉我们：

```text
dL/dw = (dL/de) * (de/dw)
      = 2e * x
```

代入 `w=3, x=2, target=10`：

```text
e = 3*2 - 10 = -4
dL/dw = 2*(-4)*2 = -16
```

和 finite difference 得到的结果一致。

In [ ]:
w = 3.0
x = 2.0
target = 10.0

prediction = w * x
error = prediction - target
analytic_derivative = 2 * error * x

print("prediction:", prediction)
print("error     :", error)
print("dL/dw     :", analytic_derivative)

这就是反向传播最核心的数学来源：

> 一个复杂计算由很多简单函数连接起来；chain rule 把局部导数一层层乘起来，把 loss 的影响传回更早的 parameter。

现在只需理解标量版本。

### 5.2 最终结论

- derivative：局部变化率 / 敏感度；
- finite difference 可以数值近似导数；
- 导数的正负给出局部变化方向；
- 对训练而言，我们关心 `d(loss)/d(parameter)`；
- chain rule 让我们能够穿过 `parameter -> prediction -> error -> loss` 的多步计算；
- 下一节 5.3：Gradient 与 Automatic Differentiation——当 parameter 不再只有一个时，MLX 怎样自动完成这些导数计算。